In [ ]:
!pip install transformers==4.41.2
!pip install peft==0.10.0
!pip install accelerate
!pip install datasets
!pip install sentencepiece

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [29]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [28]:
print(df.columns)

Index(['full_text', 'score'], dtype='object')


In [27]:
df = df[['full_text', 'score']]

df = df.dropna()

df = df.sample(20)

In [ ]:
def make_prompt(row):
    prompt = f"""
You are an expert essay grader.

Essay:
{row['full_text']}

Give a score from 1 to 6 only.
"""

    return {
        "text": prompt,
        "label": int(row['score'])
    }

data = df.apply(make_prompt, axis=1).tolist()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:

from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)

def tokenize(example):
    text = example["text"] + " Score: " + str(example["label"])

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize)

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    report_to="none",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [23]:
trainer.train()

Step,Training Loss
10,2.495200
20,2.690800


TrainOutput(global_step=20, training_loss=2.5929823875427247, metrics={'train_runtime': 4.1547, 'train_samples_per_second': 4.814, 'train_steps_per_second': 4.814, 'total_flos': 15907411722240.0, 'train_loss': 2.5929823875427247, 'epoch': 1.0})

In [ ]:
model.save_pretrained("aes-lora-model")
tokenizer.save_pretrained("aes-lora-model")

In [24]:
test_essay = """
Technology helps students learn more efficiently.
Online learning provides flexibility and access to information.
"""

prompt = f"""
You are an expert essay grader.

Essay:
{test_essay}

Give a score from 1 to 6 only.
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(outputs[0]))

<s> 
You are an expert essay grader.

Essay:

Technology helps students learn more efficiently.
Online learning provides flexibility and access to information.


Give a score from 1 to 6 only.

1. Excellent
2. Good
3. Average
4. Poor
